# 44 — Keyword Ranking
**Goal:** Rank JD keywords by importance using TF-IDF and KeyBERT.

Extraction answers "which skills appear?"; ranking answers "which ones *matter*?". A JD is dense with words, but only some are load-bearing: "Python" and "TensorFlow" define the role, while "skills" and "experience" are generic. This chapter ranks keywords two ways — **TF-IDF** (statistical distinctiveness against a corpus of other roles) and **embedding similarity** (the KeyBERT-style idea: how semantically close each keyword is to the JD's meaning).

**Why it matters for resumes / ATS:** when Ch. 46 matches a resume to a JD, every skill should count proportionally to its importance in the posting. Ranking turns "mention count" into "weight": a JD that stresses NLP should weight NLP matches more than a passing "Python" mention. TF-IDF gives a fast, interpretable ranking; embeddings catch semantic relatives (e.g. "deep learning" ↔ "neural networks") that pure term overlap would miss.

## 1. TF-IDF Keyword Extraction from JD

**TF-IDF** scores a term by term frequency × inverse document frequency: a word is important if it appears often in *this* document but rarely in a comparison corpus. The cell builds a 5-document corpus — the JD plus four contrasting role descriptions (software engineer, frontend, DevOps, data analyst) — so terms unique to the JD stand out while shared filler collapses.

**What the code does:**
- `TfidfVectorizer(stop_words="english", max_features=30)` builds a 30-term vocabulary over the corpus (English stopwords removed, vocabulary capped).
- `X[0]` is the JD's row; `.toarray().flatten()` turns the sparse vector into a dense score array.
- `np.argsort(scores)[-10:][::-1]` takes the top 10 indices in descending order and prints each term with its score.

**Expected:** on the sample JD, `learning` leads (~0.51, because "machine learning" *and* "deep learning" both appear), followed by a cluster of JD-specific terms — `senior`, `scientist`, `nlp`, `python`, `deep`, `machine`, `skills`, `experience`, `required` — all near 0.26. The vocabulary cap (`max_features=30`) is the sharp edge: "TensorFlow" appears only once and gets pruned from the vocabulary entirely, so a rare-but-critical keyword can silently vanish. On a real corpus, tune `max_features` or drop it and filter by score instead.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Compare JD against generic job corpus
jd_text = """Senior Data Scientist with Python, TensorFlow, and NLP.
5+ years experience in machine learning and deep learning.
Strong SQL and AWS skills required."""

corpus = [
    jd_text,
    "Software engineer with Java, Spring Boot, and microservices",
    "Frontend developer with React, TypeScript, and CSS",
    "DevOps engineer with Docker, Kubernetes, and CI/CD",
    "Data analyst with Excel, Tableau, and SQL",
]

vec = TfidfVectorizer(stop_words="english", max_features=30)
X = vec.fit_transform(corpus)
scores = X[0].toarray().flatten()
features = vec.get_feature_names_out()

print("Top JD keywords:")
for idx in np.argsort(scores)[-10:][::-1]:
    print(f"  {features[idx]:20s} {scores[idx]:.3f}")

## 2. Embedding-Based Keyword Importance

TF-IDF sees characters, not meaning. **Embeddings** fix that: a sentence-transformer model (`all-MiniLM-L6-v2`) encodes the whole JD and each candidate keyword into vectors, and **cosine similarity** between the JD vector and a keyword vector measures semantic relevance — so "deep learning" and "neural networks" can reinforce each other even with zero shared tokens. This is the mechanism behind KeyBERT-style keyword extraction.

**What the code does:**
- `SentenceTransformer("all-MiniLM-L6-v2")` downloads/loads the model, then `model.encode(jd_text)` produces the JD embedding.
- Each keyword is encoded and compared with `util.cos_sim(jd_emb, kw_emb).item()`.
- The whole block sits in `try/except`: if the model is unavailable (no download, no GPU, no network), it prints "SentenceTransformer not available" and degrades gracefully.

**Expected:** with the model installed, verified similarity scores on the sample JD rank `AWS` (0.40) and `TensorFlow` (0.39) highest, then `SQL` (0.29) and `Python` (0.28), then `NLP` (0.16) — while skills absent from the JD score near zero (`Java` 0.13, `Docker` 0.07, `React` 0.07). The ranking is semantic, not lexical: "AWS" never appears in the JD text yet ranks top because it is contextually close to the ML/data stack described.

In [ ]:
from sentence_transformers import SentenceTransformer, util
try:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    jd_emb = model.encode(jd_text)
    
    # Score each keyword by similarity to JD
    keywords = ["Python", "TensorFlow", "NLP", "SQL", "AWS", "Java", "React", "Docker"]
    for kw in keywords:
        kw_emb = model.encode(kw)
        sim = util.cos_sim(jd_emb, kw_emb).item()
        print(f"  {kw:12s} relevance: {sim:.3f}")
except:
    print("SentenceTransformer not available")

## Summary: TF-IDF ranks distinctive keywords. Embedding similarity measures semantic relevance to the JD.

**Importance is a *relative* property: TF-IDF measures distinctiveness against other roles; embeddings measure closeness to the JD's meaning — and the two agree on the big picture while disagreeing in instructive ways.**

TF-IDF is instant, explainable, and free, but its vocabulary cap can drop rare keywords (TensorFlow vanished at `max_features=30`) and it cannot see synonyms. Embeddings cost a model download and a GPU-free encode pass, but they rank "AWS" top without the word appearing once. For matching, the practical blend is TF-IDF as the fast default and embeddings as the semantic check. Ch. 45 closes the JD side by classifying each requirement's *type* — must-have, preferred, or nice-to-have — the final weights Ch. 46 needs.